### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_discussion import QuinexDiscussion

In [2]:
discussion = QuinexDiscussion()

### **Load Results**

In [3]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")

📂 Loaded DataFrame from '../data/quinex/df_quinex_metadata.parquet' with 31633 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex.parquet' with 3244165 rows


### **Filter data**

In [27]:
# -----------------------
# Batteries
# -----------------------
battery_patterns = {
    "Lithium-Ion Battery": [
        r"(lithium[\s-]*ion|li[\s-]*ion|liion)[\s-]*(batter(y|ies))"
    ],
    "Nickel-Iron Battery": [
        r"(nickel[\s-]*iron|ni[\s-]*fe|nife)[\s-]*(batter(y|ies))"
    ],
    "Sodium Sulfur Battery": [
        r"(sodium[\s-]*sulfur|na[\s-]*s)[\s-]*(batter(y|ies))"
    ],
    "Lithium Iron Phosphate Battery": [
        r"(lithium[\s-]*iron[\s-]*phosphate|lfp)[\s-]*(batter(y|ies))"
    ],
    "Lead Acid Battery": [
        r"(lead[\s-]*acid|la)[\s-]*(batter(y|ies))"
    ],
    "Battery": [
        r"batter(y|ies)"
    ]
}

# -----------------------
# Hydrogen
# -----------------------
hydrogen_patterns = {
    "Hydrogen": [
        r"hydrogen|\bh2\b"
    ],
    "Green Hydrogen": [
        r"green[\s-]*hydrogen"
    ],
    "Blue Hydrogen": [
        r"blue[\s-]*hydrogen"
    ],
    "Grey Hydrogen": [
        r"grey[\s-]*hydrogen"
    ],
    "Brown Hydrogen": [
        r"brown[\s-]*hydrogen"
    ],
    "Turquoise Hydrogen": [
        r"turquoise[\s-]*hydrogen"
    ],
    "Liquid Hydrogen": [
        r"liquid[\s-]*hydrogen|lh2"
    ],
    "Compressed Hydrogen": [
        r"compressed[\s-]*hydrogen|ch2"
    ],
    "Gaseous Hydrogen": [
        r"gaseous[\s-]*hydrogen|gh2"
    ]
}

# -----------------------
# PV
# -----------------------
pv_patterns = {
    "PV": [
        r"photovoltaic|pv"
    ],
    "Rooftop PV": [
        r"rooftop[\s-]*(pv|photovoltaic)"
    ],
    "Utility-scale PV": [
        r"utility[\s-]*scale[\s-]*(pv|photovoltaic)"
    ],
    "Ground-mounted PV": [
        r"ground[\s-]*mounted[\s-]*(pv|photovoltaic)"
    ],
    "Floating PV (FPV)": [
        r"floating[\s-]*(pv|fpv)"
    ],
    "Building-Integrated PV (BIPV)": [
        r"building[\s-]*integrated[\s-]*(pv|bipv)"
    ],
    "Photovoltaic-Thermal (PVT)": [
        r"photovoltaic[\s-]*thermal|pvt"
    ]
}

# -----------------------
# Wind Turbine
# -----------------------
wind_patterns = {
    "Wind Turbine": [
        r"wind[\s-]*turbine"
    ],
    "Onshore Wind Turbine": [
        r"onshore[\s-]*wind[\s-]*turbine"
    ],
    "Offshore Wind Turbine": [
        r"offshore[\s-]*wind[\s-]*turbine"
    ],
    "Floating Offshore Wind Turbine": [
        r"floating[\s-]*offshore[\s-]*wind[\s-]*turbine"
    ],
    "Small Wind Turbine": [
        r"small[\s-]*wind[\s-]*turbine"
    ],
    "Large Wind Turbine": [
        r"large[\s-]*wind[\s-]*turbine"
    ],
    "Vertical Axis Wind Turbine (VAWT)": [
        r"vertical[\s-]*axis[\s-]*wind[\s-]*turbine|vawt"
    ],
    "Horizontal Axis Wind Turbine (HAWT)": [
        r"horizontal[\s-]*axis[\s-]*wind[\s-]*turbine|hawt"
    ],
    "Airborne Wind Turbine": [
        r"airborne[\s-]*wind[\s-]*turbine"
    ],
    "Direct Drive Wind Turbine": [
        r"direct[\s-]*drive[\s-]*wind[\s-]*turbine"
    ]
}

# -----------------------
# Natural Gas
# -----------------------
natural_gas_patterns = {
    "Natural Gas": [
        r"natural[\s-]*gas|ng\b"
    ],
    "Liquefied Natural Gas (LNG)": [
        r"liquefied[\s-]*natural[\s-]*gas|lng\b"
    ],
    "Compressed Natural Gas (CNG)": [
        r"compressed[\s-]*natural[\s-]*gas|cng\b"
    ]
}

# -----------------------
# Electrolyzer
# -----------------------
electrolyzer_patterns = {
    "Electrolyzer": [
        r"electrolyzer"
    ],
    "PEM Electrolyzer": [
        r"pem[\s-]*electrolyzer|proton[\s-]*exchange[\s-]*membrane"
    ],
    "Alkaline Electrolyzer": [
        r"alkaline[\s-]*electrolyzer"
    ],
    "AEM Electrolyzer": [
        r"aem[\s-]*electrolyzer|anion[\s-]*exchange[\s-]*membrane"
    ],
    "AEC Electrolyzer": [
        r"aec[\s-]*electrolyzer|alkaline[\s-]*electrolysis"
    ],
    "CO2 Electrolyzer": [
        r"co2[\s-]*electrolyzer"
    ],
    "Solid Oxide Electrolyzer": [
        r"solid[\s-]*oxide[\s-]*electrolyzer|soe\b"
    ]
}

# -----------------------
# Hybrid
# -----------------------
hybrid_patterns = {
    "Hybrid Energy System": [
        r"hybrid[\s-]*energy[\s-]*system"
    ],
    "Hybrid Renewable Energy System (HRES)": [
        r"hybrid[\s-]*renewable[\s-]*energy[\s-]*system"
    ],
    "Hybrid Power Plant": [
        r"hybrid[\s-]*power[\s-]*plant"
    ]
}


df = df_quinex.copy()

In [28]:
import re
def map_entity(entity, regex_map):
    if entity is None:
        return None

    entity_low = entity.lower()

    # 1) Alle Matches sammeln
    matches = []

    for category, items in regex_map.items():
        for label, patterns in items.items():
            for pat in patterns:
                if re.search(pat, entity_low, flags=re.IGNORECASE):
                    matches.append((category, label))

    # Keine Matches → None
    if not matches:
        return None

    # 2) Spezialregel: Hybrid hat höchste Priorität
    hybrid_hits = [label for cat,label in matches if cat == "hybrid"]
    if hybrid_hits:
        # es gilt: wenn hybrid vorkommt → Hybrid immer nehmen
        return hybrid_hits[0]

    # 3) Prüfen auf mehrere Kategorien
    categories = set([cat for cat,label in matches])

    # Wenn mehr als eine Kategorie vorkommt → None
    if len(categories) > 1:
        return None

    # Nur 1 Kategorie → jetzt subtyp vor basistyp
    labels = [label for cat,label in matches]

    # Subtypen sind einfach die Labels außer dem generischen
    # z.B. Battery vs Lithium-Ion Battery
    # Strategie: wähle das längste Label (subtyp > basistyp)
    labels_sorted = sorted(labels, key=len, reverse=True)

    # Beispiel:
    # ["Lithium-Ion Battery", "Battery"] → Lithium-Ion Battery ist länger
    return labels_sorted[0]

In [ ]:
df["entity_mapping"] = df["entity"].apply(lambda x: map_entity(x, regex_map))

In [24]:
df_battery[df_battery["entity_mapping"] == "Nickel-Iron Battery"]["entity"].value_counts().iloc[0:50]

entity
Ni-Fe battery-based IHRES                                                                                                                                           67
Ni-Fe battery                                                                                                                                                       33
Ni-Fe battery-based IHRES with BMG                                                                                                                                  18
Ni-Fe battery-based IHRES at 80% DOD with LF strategy using the LPRAHC-based scenario                                                                               16
Ni-Fe batteries                                                                                                                                                      8
Ni-Fe battery-powered microgrid                                                                                                                               

In [ ]:
df_quinex["entity"].value_counts()[60:120]

In [ ]:
df_quinex[df_quinex["property"].str.contains("COP")]["entity"].value_counts().iloc[0:50]

In [ ]:
df_quinex[df_quinex["property"].str.contains("emission")]["property"].value_counts().iloc[0:50]

In [ ]:
df_quinex[df_quinex["entity"] == "HRES"]["property"].value_counts()

### **Results Metadata**

In [ ]:
technologies_df["property"].value_counts().iloc[0:60]

In [ ]:
technologies_df[technologies_df["property"] == "NPC"]["entity_mapping"].value_counts()

## Technologies

In [4]:
result = discussion.get_metadata(df_quinex, [kw for kws in discussion.entity_keywords_string.values() for kw in kws])

top_entities_names = result['top_entities'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)


technologies_df, mention_to_entity = discussion.perform_mapping(technologies_df, top_entities_names, entity=True, threshold=0.8)
technologies_df["entity_base_mapping"] = technologies_df["entity_mapping"].map(discussion.build_entity_reverse_mapping())
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies.parquet")

Total Statements: 427405
Number of Entities: 141383
Number of Properties: 129415
Total Statements: 68987
Number of Entities: 4261
Number of Properties: 22458
💾 Saved DataFrame to '../data/visualisation_files/df_technologies.parquet' with 68987 rows


In [5]:
import json
with open("../data/visualisation_files/mapping/technology.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_entity, f, ensure_ascii=False, indent=4)

## Properties

### Cost

In [9]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [11]:
technologies_df[technologies_df["entity_mapping"] == "Lithium-Ion Battery"]

,publication_year,DOI,classification,entity,property,quantity_text,value_normalized,unit_normalized,quantity_normalized,quote,temporal_scope,spatial_scope,entity_mapping,entity_base_mapping
1198,2010,10.3390/su2123650,"{'rational': None, 'system': 'real_world', 'type': 'observation'}",lithium ion batteries,available lithium reserves,> 30 million tons,30000000.0,"[{'exponent': 1.0, 'text': 'tons', 'unit_uri': 'http://qudt.org/vocab/unit/RT', 'year': None}]","{'quantities': [{'modifier': '>', 'unit': [{'exponent': 1.0, 'text': 'tons', 'unit_uri': 'http://qudt.org/vocab/unit/RT', 'year': None}], 'value': 30000000.0}], 'type': 'single_quantity'}","Progress has been made on lithium ion batteries; these are the power source for the currently available Tesla Roadsters [78] , and the Chevy Volt and Nissan Leaf, both slated for availability in December, 2010. This battery technology has already been rapidly adopted in ultra-lightweight electric vehicles such as electric bicycles and scooters, due to its significant advantage in energy density over other battery technologies such as lead acid and nickel metal hydride batteries. Industry reports place the available lithium reserves at over 30 million tons [79] .",None,None,Lithium-Ion Battery,battery
1201,2010,10.3390/su2123650,"{'rational': None, 'system': 'real_world', 'type': 'observation'}",lithium ion batteries,average battery capacity,= 30 kWh/vehicle,30.0,"[{'exponent': None, 'text': None, 'unit_uri': 'http://qudt.org/vocab/unit/UNITLESS', 'year': None}]","{'quantities': [{'modifier': '=', 'unit': [{'exponent': None, 'text': None, 'unit_uri': 'http://qudt.org/vocab/unit/UNITLESS', 'year': None}], 'value': 30.0}], 'type': 'single_quantity'}","Progress has been made on lithium ion batteries; these are the power source for the currently available Tesla Roadsters [78] , and the Chevy Volt and Nissan Leaf, both slated for availability in December, 2010. This battery technology has already been rapidly adopted in ultra-lightweight electric vehicles such as electric bicycles and scooters, due to its significant advantage in energy density over other battery technologies such as lead acid and nickel metal hydride batteries. Industry reports place the available lithium reserves at over 30 million tons [79] . With approximately 250 million total motor vehicles currently in use in the US [80], it would require roughly 1.35 million tons of lithium to electrify US transportation, assuming an average battery capacity of 30 kWh/vehicle (the Nissan Leaf has a 24 kWh battery), or around 4.",{'year': 2010.0},None,Lithium-Ion Battery,battery
1202,2010,10.3390/su2123650,"{'rational': None, 'system': 'real_world', 'type': 'observation'}",lithium ion batteries,energy storage capacity,= 24 kWh,24.0,"[{'exponent': 1.0, 'text': 'kWh', 'unit_uri': 'http://qudt.org/vocab/unit/KiloW-HR', 'year': None}]","{'quantities': [{'modifier': '=', 'unit': [{'exponent': 1.0, 'text': 'kWh', 'unit_uri': 'http://qudt.org/vocab/unit/KiloW-HR', 'year': None}], 'value': 24.0}], 'type': 'single_quantity'}","Progress has been made on lithium ion batteries; these are the power source for the currently available Tesla Roadsters [78] , and the Chevy Volt and Nissan Leaf, both slated for availability in December, 2010. This battery technology has already been rapidly adopted in ultra-lightweight electric vehicles such as electric bicycles and scooters, due to its significant advantage in energy density over other battery technologies such as lead acid and nickel metal hydride batteries. Industry reports place the available lithium reserves at over 30 million tons [79] . With approximately 250 million total motor vehicles currently in use in the US [80], it would require roughly 1.35 million tons of lithium to electrify US transportation, assuming an average battery capacity of 30 kWh/vehicle (the Nissan Leaf has a 24 kWh battery), or around 4.",{'year': 2010.0},None,Lithium-Ion Battery,battery
1203,2010,10.3390/su2123650,"{'ration

In [7]:
result = discussion.get_metadata(technologies_df, property_keywords=discussion.property_keywords_string["cost"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="cost", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)


discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies_cost.parquet")

Total Statements: 9728
Number of Entities: 1153
Number of Properties: 2617
Total Statements: 2147
Number of Entities: 433
Number of Properties: 83
💾 Saved DataFrame to '../data/visualisation_files/df_technologies_cost.parquet' with 2147 rows


In [8]:
import json
with open("../data/visualisation_files/mapping/cost.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)

### Lifetime

In [9]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [10]:
result = discussion.get_metadata(technologies_df, property_keywords=discussion.property_keywords_string["lifetime"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="lifetime", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies_lifetime.parquet")

Total Statements: 2880
Number of Entities: 464
Number of Properties: 658
Total Statements: 1173
Number of Entities: 238
Number of Properties: 12
💾 Saved DataFrame to '../data/visualisation_files/df_technologies_lifetime.parquet' with 1173 rows


In [11]:
import json
with open("../data/visualisation_files/mapping/lifetime.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)

### Capacity

In [12]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [13]:
result = discussion.get_metadata(technologies_df, property_keywords=discussion.property_keywords_string["capacity"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="capacity", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies_capacity.parquet")

Total Statements: 10371
Number of Entities: 1068
Number of Properties: 1125
Total Statements: 6710
Number of Entities: 795
Number of Properties: 91
💾 Saved DataFrame to '../data/visualisation_files/df_technologies_capacity.parquet' with 6710 rows


In [14]:
import json
with open("../data/visualisation_files/mapping/capacity.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)

### Efficiency

In [15]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [16]:
result = discussion.get_metadata(technologies_df, property_keywords=discussion.property_keywords_string["efficiency"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="efficiency", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies_efficiency.parquet")

Total Statements: 3066
Number of Entities: 609
Number of Properties: 1037
Total Statements: 988
Number of Entities: 291
Number of Properties: 60
💾 Saved DataFrame to '../data/visualisation_files/df_technologies_efficiency.parquet' with 988 rows


In [17]:
import json
with open("../data/visualisation_files/mapping/efficiency.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)

### LPSP

In [18]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")

In [19]:
result = discussion.get_metadata(technologies_df, property_keywords=discussion.property_keywords_string["LPSP"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="LPSP", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_technologies_lpsp.parquet")

Total Statements: 212
Number of Entities: 52
Number of Properties: 38
Total Statements: 69
Number of Entities: 20
Number of Properties: 13
💾 Saved DataFrame to '../data/visualisation_files/df_technologies_lpsp.parquet' with 69 rows


In [20]:
import json
with open("../data/visualisation_files/mapping/lpsp.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)

### Emission

In [21]:
result = discussion.get_metadata(df_quinex, property_keywords=discussion.property_keywords_string["emission"])

top_properties_names = result['top_properties'].index.tolist()
technologies_df = result["filtered_df"]
discussion.print_df_sizes(technologies_df)

technologies_df, mention_to_property = discussion.perform_mapping(technologies_df, top_properties_names, canonical_name="emission", property=True, threshold=0.8)
discussion.print_df_sizes(technologies_df)

discussion.save_df(technologies_df, "../data/visualisation_files/df_emission.parquet")

Total Statements: 87123
Number of Entities: 36772
Number of Properties: 19570
Total Statements: 22895
Number of Entities: 11819
Number of Properties: 966
💾 Saved DataFrame to '../data/visualisation_files/df_emission.parquet' with 22895 rows


In [22]:
import json
with open("../data/visualisation_files/mapping/emission.json", "w", encoding="utf-8") as f:
    json.dump(mention_to_property, f, ensure_ascii=False, indent=4)